# Random Forests, Explained Simply
Many decision trees, each a little different, voting together. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A property website has 1,000 house sales: size, bedrooms, age, distance to the train station, garden, suburb rating (1 to 5), and price in thousands of dollars.

### 1,000 house sales
- `houses.csv`: 1,000 sales: size_m2, bedrooms, age_years, km_to_station, garden (1 = yes), suburb_rating, price_k (thousands of dollars)

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "random-forests/houses.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Train and test sets
- `test_size=0.25`: a quarter of the houses kept for testing

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="price_k")
y = df["price_k"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
print(len(X_train), "training houses,", len(X_test), "test houses")

## 2. One tree is unstable
A single deep tree overfits, and trees trained on slightly different data can disagree a lot.

### A single tree
- `DecisionTreeRegressor`: a decision tree that predicts a number: each leaf gives the average price of its training houses
- `score`: R squared: 1 is perfect, 0 is no better than guessing the average
- `the output`: perfect on training, much worse on test: overfitting

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("train R squared:", round(tree.score(X_train, y_train), 3))
print("test R squared: ", round(tree.score(X_test, y_test), 3))

### Five trees, five different answers
- `X_test.iloc[[6]]`: one test house
- `choice(len(X_train), len(X_train))`: a random sample of the training houses, with replacement (a bootstrap sample)
- `the output`: each tree, trained on a slightly different sample, gives a different price

In [ ]:
import numpy as np

house = X_test.iloc[[6]]
print("real price:", y_test.iloc[6])
for seed in range(5):
    rows = np.random.default_rng(seed).choice(len(X_train), len(X_train))
    t = DecisionTreeRegressor(random_state=0)
    t.fit(X_train.iloc[rows], y_train.iloc[rows])
    print(f"tree {seed + 1}: {t.predict(house)[0]:.0f}")

## 3. A random forest
Train many trees, each on its own bootstrap sample and looking at a random subset of inputs at each split. Average their predictions.

### Train a forest
- `n_estimators=200`: 200 trees
- `oob_score=True`: also score each tree on the houses its bootstrap sample left out
- `n_jobs=-1`: use every processor core, to train trees in parallel
- `the output`: far better than one tree; the out-of-bag score agrees with the test score

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=200, oob_score=True,
                               random_state=0, n_jobs=-1)
forest.fit(X_train, y_train)
print("test R squared:  ", round(forest.score(X_test, y_test), 3))
print("out-of-bag score:", round(forest.oob_score_, 3))

### The forest's answer for one house
- `forest.estimators_`: the 200 individual trees
- `all_trees.mean()`: the forest's prediction is just the average
- `np.percentile(..., [10, 90])`: the range covering the middle 80% of the tree predictions

In [ ]:
all_trees = np.array([t.predict(house.values)[0] for t in forest.estimators_])
print("real price:          ", y_test.iloc[6])
print("forest (average):    ", round(all_trees.mean()))
print("middle 80% of trees: ", np.percentile(all_trees, [10, 90]).round())

### How many trees?
- `sizes`: forests from 1 tree to 200
- `the curve`: rises fast, then levels off: more trees never hurt, they just stop helping

In [ ]:
import matplotlib.pyplot as plt

sizes = [1, 2, 5, 10, 25, 50, 100, 200]
scores = [RandomForestRegressor(n_estimators=n, random_state=0, n_jobs=-1)
          .fit(X_train, y_train).score(X_test, y_test) for n in sizes]
plt.plot(sizes, scores, marker="o")
plt.xscale("log"); plt.xlabel("number of trees"); plt.ylabel("test R squared")
plt.show()

## 4. Which inputs matter?
Random forests average the feature importances of all their trees.

### Feature importance
- `feature_importances_`: how much each input reduced the error, averaged over all 200 trees

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
imp.plot.barh()
plt.xlabel("importance"); plt.title("What the forest relies on")
plt.show()

## 5. Mini project: a price estimate with a range
Use the spread of the trees as a rough range.

### Estimate a house price
- `**house`: accept the house's facts by name
- `[X.columns]`: put the columns in the same order as training
- `np.percentile`: the range most trees agree on

In [ ]:
def estimate(**house):
    h = pd.DataFrame([house])[X.columns]
    preds = np.array([t.predict(h.values)[0] for t in forest.estimators_])
    low, high = np.percentile(preds, [10, 90])
    return f"about ${preds.mean():.0f}k (most trees: ${low:.0f}k to ${high:.0f}k)"

print(estimate(size_m2=120, bedrooms=3, age_years=25, km_to_station=0.8,
               garden=1, suburb_rating=4))

## Try it yourself
1. Price a 180 m2, 4-bedroom, 10-year-old house, 1 km from the station, with a garden, in a rating-5 suburb.
2. Try max_features=1.0 (every tree sees every input at every split). Does the score change?
3. Try n_estimators=10 and n_estimators=500. How much does the score change?